# **Session 6**

## Exploratory Data Analysis (EDA) - Summary Statistics, Categorical and Continuous Variables

In Session 5 we loaded the data, looked at its structure, used `df.describe()` and drew our first scatter plot.
In this session we go much deeper. We will **explore the data properly**, the way a real data analyst does before building any model.

### Session Roadmap (about 2 hours)

| Part | Topic | Approx. Time |
|---|---|---|
| 1 | Setup and quick recap | 10 min |
| 2 | Data quality checks (missing values, duplicates, placeholders) | 15 min |
| 3 | Categorical vs Continuous variables | 10 min |
| 4 | Summary statistics for continuous variables | 20 min |
| 5 | Summary statistics for categorical variables | 15 min |
| 6 | Univariate visualization (one variable at a time) | 15 min |
| 7 | Outlier detection | 10 min |
| 8 | Bivariate analysis (two variables together) | 20 min |
| 9 | Multivariate analysis and feature creation | 10 min |
| 10 | Reusable EDA function, insights and practice | 5 min |

### What is EDA?

**EDA (Exploratory Data Analysis)** means looking at the data from every angle **before** doing any modelling. The goals are:

1. Understand what each column contains.
2. Find problems (missing values, wrong types, duplicates, outliers).
3. Understand the shape (distribution) of each variable.
4. Find relationships between variables.
5. Form questions and hypotheses that we can test later.

## Part 1: Setup and Quick Recap

We start exactly like Session 5: mount Google Drive and load the data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls "/content/drive/MyDrive/Colab Notebooks/Data/"

import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Data/academic_data.csv")

### Importing the libraries we need today

| Library | Short name | Used for |
|---|---|---|
| `pandas` | `pd` | Tables (DataFrames), grouping, summaries (already imported above) |
| `numpy` | `np` | Mathematical operations on numbers |
| `matplotlib.pyplot` | `plt` | Basic plotting (we used it in Session 5) |
| `seaborn` | `sns` | Prettier and more powerful statistical plots built on top of matplotlib |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Global look and feel for all plots
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

# Show 2 decimal places for floats so tables are easy to read
pd.set_option("display.float_format", lambda x: f"{x:.2f}")
pd.set_option("display.max_columns", 50)

**Explanation of the settings**

- `sns.set_theme(style="whitegrid")` applies a clean white background with light grid lines to every plot.
- `plt.rcParams["figure.figsize"] = (8, 5)` sets the default plot size (width 8 inches, height 5 inches).
- `pd.set_option("display.float_format", ...)` makes pandas show numbers with 2 decimal places.
- `pd.set_option("display.max_columns", 50)` makes sure pandas does not hide columns when a table is wide.

### Quick recap of Session 5

In [ ]:
print("Shape (rows, columns):", df.shape)
df.head()

In [ ]:
df.info()

**What we already know from Session 5**

- 100 students and 14 columns.
- `ai_tool` has only 93 non-null values, so 7 values are missing.
- 8 columns are text (`object`), 4 are decimal (`float64`) and 2 are whole numbers (`int64`).

Today we will find out much more than this.

## Part 2: Data Quality Checks

Before calculating anything, we must make sure the data is trustworthy. If the data is wrong, every result will be wrong too.
We check four things:

1. Missing values
2. Duplicate rows
3. Placeholder values (such as `-`) that look like data but actually mean "nothing"
4. Wrong or unexpected values

### 2.1 Missing Values

`df.isnull()` returns `True` wherever a value is missing. Adding `.sum()` counts the `True` values in each column.

In [ ]:
missing_count = df.isnull().sum()
missing_percent = (df.isnull().mean() * 100).round(2)

missing_table = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_percent
})

# Show only the columns that actually have missing values
missing_table[missing_table["missing_count"] > 0]

**Explanation**

- `df.isnull().sum()` gives the number of missing cells per column.
- `df.isnull().mean()` works because `True` counts as 1 and `False` as 0, so the mean is the **fraction** of missing values. Multiplying by 100 gives a percentage.
- `pd.DataFrame({...})` combines both results into one neat table.
- `missing_table[missing_table["missing_count"] > 0]` is **filtering**: it keeps only the rows where the condition is `True`.

**Result:** only `ai_tool` has missing values (7 out of 100, which is 7 percent).

### 2.2 Why is `ai_tool` missing?

Missing does not always mean "error". Sometimes it carries meaning. Let us compare the AI usage hours of students whose `ai_tool` is missing with the rest.

In [ ]:
# Rows where ai_tool is missing
df[df["ai_tool"].isnull()][["student_id", "ai_usage_hours", "ai_tool", "marks"]]

In [ ]:
# Compare average AI usage hours: missing tool vs. tool present
df.groupby(df["ai_tool"].isnull())["ai_usage_hours"].agg(["count", "mean", "min", "max"])

**Explanation**

- `df["ai_tool"].isnull()` creates a True/False column. We use it to **group** students into two groups: `True` (no tool recorded) and `False` (tool recorded).
- `.agg(["count", "mean", "min", "max"])` calculates several statistics in one go.

If the students with a missing tool also have very low AI usage hours, it is reasonable to say they simply **do not use any AI tool**. In that case, replacing the missing value with the label `"No Tool"` is better than deleting those students.

### 2.3 Duplicate Rows and Unique IDs

In [ ]:
print("Fully duplicated rows        :", df.duplicated().sum())
print("Is student_id unique?        :", df["student_id"].is_unique)
print("Number of repeated names     :", df["name"].duplicated().sum())

**Explanation**

- `df.duplicated()` marks a row `True` if the **entire row** is an exact copy of an earlier row.
- `.is_unique` checks that every value in the column is different. A student ID must be unique.
- Names can repeat naturally (two different students can both be called "Ishaan Das"), so repeated names are **not** an error. The ID is what identifies a student.

### 2.4 Placeholder Values

Look at `complaint_type`. In Session 5 we saw that its most common value (`top`) is `-` with a frequency of 91.
A dash is not a real complaint type. It is a **placeholder** meaning "no complaint". Pandas does not see it as missing, so `isnull()` could not find it.

In [ ]:
df["complaint_type"].value_counts()

In [ ]:
# Does the placeholder match exactly with students who made no complaint?
pd.crosstab(df["complaint"], df["complaint_type"])

**Explanation**

- `value_counts()` counts how many times each different value appears.
- `pd.crosstab(a, b)` builds a **cross table**: rows are values of `a`, columns are values of `b`, and each cell counts the students having that combination.

The table confirms the pattern: students with `complaint = No` have `complaint_type = -`. So `-` really means "no complaint".

### 2.5 Creating a Clean Copy

We never modify the original `df`. We create a copy called `df_clean` and fix problems there.
This way, if we make a mistake, the original data is still safe.

In [ ]:
df_clean = df.copy()

# 1) Missing AI tool means the student uses no AI tool
df_clean["ai_tool"] = df_clean["ai_tool"].fillna("No Tool")

# 2) Replace the dash placeholder with a meaningful label
df_clean["complaint_type"] = df_clean["complaint_type"].replace("-", "No Complaint")

# Verify
print("Missing values left:", df_clean.isnull().sum().sum())
print()
print(df_clean["ai_tool"].value_counts())
print()
print(df_clean["complaint_type"].value_counts())

**Explanation**

- `df.copy()` creates an independent copy. Without `.copy()`, changes to `df_clean` could also change `df`.
- `fillna("No Tool")` replaces every missing value with the text `No Tool`.
- `replace("-", "No Complaint")` swaps the placeholder for a clear label.
- `.sum().sum()` adds up the missing counts of all columns into one single number (it should be 0 now).

## Part 3: Categorical vs Continuous Variables

Every analysis depends on the **type** of variable. We use different statistics and different graphs for different types.

### The Variable Family Tree

| Type | Sub-type | Meaning | Examples in our data |
|---|---|---|---|
| **Categorical** | Nominal | Categories with **no order** | `gender`, `section`, `ai_tool`, `complaint`, `complaint_type` |
| **Categorical** | Ordinal | Categories **with a natural order** | `grade` (A > B > C > D > F), `year` (1st to 4th) |
| **Continuous (Numerical)** | Continuous | Can take any value in a range, including decimals | `study_hours`, `attendance`, `ai_usage_hours`, `marks` |
| **Continuous (Numerical)** | Discrete | Whole-number counts | `age` |
| **Identifier** | - | Just a label, not useful for statistics | `student_id`, `name` |

### Important point

A column stored as a number is **not always** continuous. `year` is stored as `int64`, but the values 1, 2, 3, 4 are really **categories** (first year, second year ...). Averaging years makes no sense.
Always decide the variable type by **meaning**, not only by the data type.

In [ ]:
# Quick look: how many different values does each column have?
unique_table = pd.DataFrame({
    "dtype": df_clean.dtypes,
    "unique_values": df_clean.nunique()
})
unique_table

**Explanation**

- `df_clean.dtypes` gives the data type of each column.
- `df_clean.nunique()` gives the number of **different** values in each column.

A column with very few unique values (like `gender` with 2, `year` with 4) is usually categorical.
A column with many unique values (like `marks`) is usually continuous.

### Defining the column groups

We store the column names in lists. This saves us from typing the names again and again.

In [ ]:
id_cols          = ["student_id", "name"]
categorical_cols = ["gender", "section", "year", "ai_tool", "grade", "complaint", "complaint_type"]
continuous_cols  = ["age", "study_hours", "attendance", "ai_usage_hours", "marks"]

print("Identifier columns :", id_cols)
print("Categorical columns:", categorical_cols)
print("Continuous columns :", continuous_cols)
print()
print("Total columns accounted for:", len(id_cols) + len(categorical_cols) + len(continuous_cols), "out of", df_clean.shape[1])

### Telling pandas that `grade` has an order

By default pandas sorts text alphabetically, which happens to work for A to F. But explicitly marking `grade` as an **ordered categorical** makes our tables and graphs always follow the correct order (best grade first).

In [ ]:
grade_order = ["A", "B", "C", "D", "F"]

df_clean["grade"] = pd.Categorical(df_clean["grade"], categories=grade_order, ordered=True)

print(df_clean["grade"].dtype)
df_clean["grade"].value_counts(sort=False)

**Explanation**

- `pd.Categorical(..., categories=grade_order, ordered=True)` tells pandas the exact list of categories and that their order is meaningful.
- `value_counts(sort=False)` keeps the categories in our defined order (A, B, C, D, F) instead of sorting by count.

## Part 4: Summary Statistics for Continuous Variables

Summary statistics describe a whole column using a few numbers. They fall into four families:

| Family | Question it answers | Measures |
|---|---|---|
| **Central tendency** | What is a typical value? | mean, median, mode |
| **Spread (dispersion)** | How different are the values from each other? | range, variance, standard deviation, IQR, coefficient of variation |
| **Position** | Where does a value stand compared to others? | minimum, maximum, quartiles, percentiles |
| **Shape** | What does the distribution look like? | skewness, kurtosis |

### 4.1 Starting point: `describe()`

We already used this in Session 5.

In [ ]:
df_clean[continuous_cols].describe()

In [ ]:
# .T means "transpose": rows and columns are swapped, which is easier to read when we have many statistics
df_clean[continuous_cols].describe().T

### 4.2 Measures of Central Tendency

| Measure | Definition | Pandas method | When it is best |
|---|---|---|---|
| **Mean** | Sum of values divided by count | `.mean()` | Data without extreme values |
| **Median** | Middle value after sorting | `.median()` | Data with outliers or skewed data |
| **Mode** | Most frequent value | `.mode()` | Discrete data or categories |

In [ ]:
print("Mean   of marks :", round(df_clean["marks"].mean(), 2))
print("Median of marks :", round(df_clean["marks"].median(), 2))
print("Mode   of marks :", df_clean["marks"].mode().tolist())

**Explanation**

- `.mode()` returns a **Series** because there can be more than one mode (a tie). So we convert it with `.tolist()` to see all of them.
- If the mean and the median are close, the distribution is roughly symmetric. If they are far apart, the data is skewed.

Let us calculate all three for every continuous column at once.

In [ ]:
central = pd.DataFrame({
    "mean":   df_clean[continuous_cols].mean(),
    "median": df_clean[continuous_cols].median(),
    "mode":   df_clean[continuous_cols].mode().iloc[0]   # first mode of each column
})
central["mean - median"] = central["mean"] - central["median"]
central

**Explanation**

- `.mode().iloc[0]` takes the first row of the mode table, so we get one mode per column.
- `mean - median` is a quick skewness hint:
  - Close to 0 means the data is roughly symmetric.
  - Clearly positive means a few large values are pulling the mean up (right skew).
  - Clearly negative means a few small values are pulling the mean down (left skew).

### 4.3 Measures of Spread

Two columns can have the same mean but be completely different. Example: marks of `[50, 50, 50]` and `[0, 50, 100]` both have mean 50, but the second is much more spread out.

| Measure | Formula / Meaning | Pandas method |
|---|---|---|
| **Range** | max - min | `.max() - .min()` |
| **Variance** | Average of squared distances from the mean | `.var()` |
| **Standard Deviation (std)** | Square root of variance (same unit as the data) | `.std()` |
| **IQR** | Q3 - Q1, the spread of the middle 50 percent of data | `.quantile(0.75) - .quantile(0.25)` |
| **Coefficient of Variation (CV)** | (std / mean) x 100, the spread as a percentage of the mean. Lets us compare columns with different units | calculated |

In [ ]:
spread = pd.DataFrame({
    "min":      df_clean[continuous_cols].min(),
    "max":      df_clean[continuous_cols].max(),
    "range":    df_clean[continuous_cols].max() - df_clean[continuous_cols].min(),
    "variance": df_clean[continuous_cols].var(),
    "std":      df_clean[continuous_cols].std(),
    "Q1":       df_clean[continuous_cols].quantile(0.25),
    "Q3":       df_clean[continuous_cols].quantile(0.75)
})
spread["IQR"] = spread["Q3"] - spread["Q1"]
spread["CV_%"] = (spread["std"] / df_clean[continuous_cols].mean()) * 100
spread

**Explanation**

- Each line builds one column of the new table. The index of every result is the list of column names, so pandas aligns them automatically.
- The `std` of `marks` was 17.76 in Session 5. That means a typical student's marks are about 17.76 away from the average.
- **Why CV?** `attendance` (in percent) and `study_hours` (in hours) have different units, so comparing their `std` directly is unfair. CV converts spread into a percentage of the mean, so we can compare them. A higher CV means the column is more variable relative to its average.

### 4.4 Percentiles (Measures of Position)

A **percentile** tells us the value below which a given percent of the data falls. The 90th percentile of marks is the mark that 90 percent of students are below.

- 25th percentile = Q1 (first quartile)
- 50th percentile = Q2 = median
- 75th percentile = Q3 (third quartile)

In [ ]:
percentiles = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
df_clean[continuous_cols].quantile(percentiles)

**Explanation**

`quantile()` accepts a list of fractions between 0 and 1. The result is a table: one row for each fraction and one column for each variable.
For example, the row `0.90` in the `marks` column is the 90th percentile of marks.

### 4.5 Shape of the Distribution: Skewness and Kurtosis

**Skewness** measures the **lack of symmetry**.

| Skewness value | Meaning | Mean vs Median | Picture |
|---|---|---|---|
| About 0 | Symmetric | Mean is about equal to median | Balanced bell |
| Positive (> 0.5) | Right-skewed (long tail on the right) | Mean > Median | Most values are small, a few are very large |
| Negative (< -0.5) | Left-skewed (long tail on the left) | Mean < Median | Most values are large, a few are very small |

**Kurtosis** (as calculated by pandas, also called excess kurtosis) measures how heavy the tails are.

| Kurtosis value | Meaning |
|---|---|
| About 0 | Similar to a normal bell curve |
| Positive | Heavy tails and a sharp peak (more extreme values than normal) |
| Negative | Light tails and a flat top (fewer extreme values than normal) |

In [ ]:
shape = pd.DataFrame({
    "skewness": df_clean[continuous_cols].skew(),
    "kurtosis": df_clean[continuous_cols].kurt()
})

def describe_skew(value):
    if value > 0.5:
        return "Right-skewed"
    elif value < -0.5:
        return "Left-skewed"
    else:
        return "Roughly symmetric"

shape["skew_interpretation"] = shape["skewness"].apply(describe_skew)
shape

**Explanation**

- `.skew()` and `.kurt()` calculate the two shape measures for each column.
- We wrote a small **function** `describe_skew()` that converts the number into a readable label. `.apply(function)` runs that function on every value of the column.
- The cut-offs 0.5 and -0.5 are a common rule of thumb, not a strict law.

### 4.6 One Complete Summary Table

Now we combine everything into one professional summary table. This is the kind of table you would put in a report.

In [ ]:
def continuous_summary(data, columns):
    """Return a full summary-statistics table for the given continuous columns."""
    d = data[columns]
    out = pd.DataFrame({
        "count":    d.count(),
        "mean":     d.mean(),
        "median":   d.median(),
        "std":      d.std(),
        "min":      d.min(),
        "Q1":       d.quantile(0.25),
        "Q3":       d.quantile(0.75),
        "max":      d.max(),
    })
    out["range"]    = out["max"] - out["min"]
    out["IQR"]      = out["Q3"] - out["Q1"]
    out["CV_%"]     = out["std"] / out["mean"] * 100
    out["skewness"] = d.skew()
    out["kurtosis"] = d.kurt()
    return out.round(2)

continuous_summary(df_clean, continuous_cols)

**Explanation**

- `def continuous_summary(data, columns):` creates a **reusable function**. We can call it again later for any DataFrame and any list of columns.
- The text inside triple quotes at the top is a **docstring**. It documents what the function does.
- `return out.round(2)` hands the finished table back, rounded to 2 decimals.

## Part 5: Summary Statistics for Categorical Variables

For categories, mean and standard deviation make no sense. Instead we use **counts** and **proportions**.

| Measure | Meaning | Pandas method |
|---|---|---|
| **Frequency** | How many times each category appears | `value_counts()` |
| **Relative frequency (percent)** | Share of each category | `value_counts(normalize=True)` |
| **Mode** | Most common category | `.mode()` |
| **Number of categories** | How many different categories exist | `.nunique()` |

### 5.1 `describe()` for categorical columns

In [ ]:
df_clean[categorical_cols].astype("object").describe()

**Explanation**

- `astype("object")` converts the columns to plain text type so that `describe()` gives us `count`, `unique`, `top` and `freq`.
- `unique` is the number of different categories, `top` is the mode, and `freq` is how many times the mode appears.

### 5.2 Frequency table of a single variable

In [ ]:
counts  = df_clean["gender"].value_counts()
percent = df_clean["gender"].value_counts(normalize=True) * 100

pd.DataFrame({"count": counts, "percent": percent.round(1)})

**Explanation**

- `value_counts()` gives the **count** of each category.
- `value_counts(normalize=True)` gives the **fraction** of each category. Multiplying by 100 gives percentages.
- We joined both into one DataFrame to see counts and percentages side by side.

### 5.3 A reusable function for every categorical column

In [ ]:
def frequency_table(data, column):
    """Return counts and percentages for one categorical column."""
    counts  = data[column].value_counts(sort=False) if str(data[column].dtype) == "category" else data[column].value_counts()
    percent = (counts / counts.sum() * 100).round(1)
    table   = pd.DataFrame({"count": counts, "percent": percent})
    table["cumulative_percent"] = table["percent"].cumsum().round(1)
    return table

for col in categorical_cols:
    print("=" * 45)
    print("Variable:", col)
    print("=" * 45)
    display(frequency_table(df_clean, col))
    print()

**Explanation**

- The function builds a table with `count`, `percent` and `cumulative_percent` (running total of the percentages).
- If the column is an ordered category (like `grade`) we use `sort=False` so that A, B, C, D, F stay in order.
- The `for` loop calls the function for every categorical column. `display()` shows a table nicely inside a loop (a normal expression would only show the last one).

**Things to look for**

1. **Dominant category:** if one category has most of the data (like `complaint = No`), the variable is **imbalanced**.
2. **Rare categories:** categories with very few rows are hard to analyse reliably.
3. **Cumulative percent:** useful for ordered variables, for example "what percent of students got grade C or better?"

### 5.4 Mode and Number of Categories

In [ ]:
cat_overview = pd.DataFrame({
    "unique_categories": df_clean[categorical_cols].nunique(),
    "mode": [df_clean[c].mode()[0] for c in categorical_cols],
    "mode_count": [df_clean[c].value_counts().max() for c in categorical_cols]
})
cat_overview["mode_percent"] = (cat_overview["mode_count"] / len(df_clean) * 100).round(1)
cat_overview

**Explanation**

- The part inside `[... for c in categorical_cols]` is a **list comprehension**. It is a short way of writing a loop that builds a list. Here it finds the mode of each column.
- `mode_percent` tells us how dominant the most common category is. A very high value (close to 100) means the variable has almost no variety.

## Part 6: Univariate Visualization (One Variable at a Time)

Numbers can hide patterns. Graphs show the **shape** of the data.

### Which graph for which variable?

| Variable type | Best graphs |
|---|---|
| Continuous | Histogram, KDE (density) curve, Boxplot |
| Categorical | Bar chart (count plot), Pie chart (only for few categories) |

### 6.1 Histogram

A **histogram** divides the range of a continuous variable into **bins** (intervals) and counts how many values fall into each bin. The height of each bar shows the count.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df_clean, x="marks", bins=10)
plt.title("Distribution of Marks")
plt.xlabel("Marks")
plt.ylabel("Number of Students")
plt.show()

**Explanation**

- `sns.histplot(data=df_clean, x="marks", bins=10)` draws the histogram. `bins=10` splits the range into 10 intervals.
- Too few bins hide details, too many bins make the graph noisy. Try different values (5, 10, 20) and see the difference.

### Adding a KDE curve, mean line and median line

A **KDE (Kernel Density Estimate)** is a smooth curve that estimates the shape of the distribution.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df_clean, x="marks", bins=12, kde=True)

plt.axvline(df_clean["marks"].mean(),   color="red",   linestyle="--", label="Mean")
plt.axvline(df_clean["marks"].median(), color="green", linestyle="-",  label="Median")

plt.title("Marks: Histogram with KDE, Mean and Median")
plt.legend()
plt.show()

**Explanation**

- `kde=True` adds the smooth density curve on top of the bars.
- `plt.axvline(x, ...)` draws a vertical line at position `x`. We draw one at the mean and one at the median.
- If the two lines are almost on top of each other, the distribution is roughly symmetric.

### Histograms of all continuous variables together

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(continuous_cols):
    sns.histplot(data=df_clean, x=col, bins=12, kde=True, ax=axes[i])
    axes[i].axvline(df_clean[col].mean(),   color="red",   linestyle="--")
    axes[i].axvline(df_clean[col].median(), color="green", linestyle="-")
    axes[i].set_title(f"Distribution of {col}")

# We have 5 variables but 6 slots, so hide the empty sixth slot
axes[5].set_visible(False)

plt.tight_layout()
plt.show()

**Explanation**

- `plt.subplots(2, 3)` creates a grid of 2 rows x 3 columns of plots. `axes` holds all the small plot areas.
- `axes.flatten()` turns the 2D grid into a simple list so that we can loop over it.
- `enumerate(continuous_cols)` gives us both the position `i` and the column name `col` in the loop.
- `ax=axes[i]` tells seaborn which small plot area to draw on.
- `plt.tight_layout()` prevents titles and labels from overlapping.
- The red dashed line is the mean and the green solid line is the median.

### 6.2 Boxplot

A **boxplot** summarises the distribution using five numbers and also highlights possible outliers.

| Part of the boxplot | Meaning |
|---|---|
| Line inside the box | Median (50th percentile) |
| Bottom of the box | Q1 (25th percentile) |
| Top of the box | Q3 (75th percentile) |
| Height of the box | IQR (Q3 - Q1) |
| Whiskers | Extend to the furthest value within 1.5 x IQR from the box |
| Dots beyond the whiskers | Possible **outliers** |

In [ ]:
plt.figure(figsize=(6, 5))
sns.boxplot(data=df_clean, y="marks")
plt.title("Boxplot of Marks")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))

for i, col in enumerate(["study_hours", "attendance", "ai_usage_hours", "marks"]):
    sns.boxplot(data=df_clean, y=col, ax=axes[i])
    axes[i].set_title(col)

plt.tight_layout()
plt.show()

**Explanation**

We draw each variable in its own small plot. We do not put them in the same plot because the variables have different scales (attendance is up to 100, ai_usage_hours is up to about 14), and the smaller ones would look flat.

### 6.3 Bar Chart for Categorical Variables

A **bar chart** (count plot) shows the frequency of each category. Each bar is one category and the height is its count.

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=df_clean, x="gender", color="steelblue")
plt.title("Number of Students by Gender")
plt.xlabel("Gender")
plt.ylabel("Count")
plt.show()

**Explanation**

`sns.countplot()` counts the rows in each category automatically, so we do not need to run `value_counts()` first.

### Bar charts for all categorical variables

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    if col == "year":
        order = sorted(df_clean[col].unique())             # 1, 2, 3, 4 in natural order
    elif col == "grade":
        order = grade_order                                # A, B, C, D, F
    else:
        order = df_clean[col].value_counts().index         # largest category first

    sns.countplot(data=df_clean, x=col, order=order, color="steelblue", ax=axes[i])
    axes[i].set_title(f"Count of {col}")
    axes[i].tick_params(axis="x", rotation=30)

axes[7].set_visible(False)   # 7 variables, 8 slots

plt.tight_layout()
plt.show()

**Explanation**

- `order=` controls the order of the bars. For ordered variables (`year`, `grade`) we use the natural order. For others we sort from the biggest to the smallest, which is easier to read.
- `tick_params(axis="x", rotation=30)` rotates the labels on the x-axis so long names do not overlap.
- `set_visible(False)` hides the unused eighth plot.

### 6.4 Pie Chart

A pie chart shows each category as a **share of the whole**. Use it only when there are **few categories (2 to 5)**. With many categories, a bar chart is clearer.

In [ ]:
gender_counts = df_clean["gender"].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(gender_counts, labels=gender_counts.index, autopct="%1.1f%%", startangle=90)
plt.title("Gender Share of Students")
plt.show()

**Explanation**

- `plt.pie(values, labels=...)` draws the pie.
- `autopct="%1.1f%%"` prints each slice's percentage with one decimal place.
- `startangle=90` rotates the start of the first slice so the chart looks tidy.

## Part 7: Outlier Detection

An **outlier** is a value that is very different from most of the data. Outliers can be genuine (a really brilliant student) or mistakes (a typing error). We must **find** them first and then **decide** what to do.

### The IQR Rule

A value is a possible outlier if:

- it is **below** `Q1 - 1.5 x IQR` (lower fence), or
- it is **above** `Q3 + 1.5 x IQR` (upper fence).

This is exactly the rule a boxplot uses to draw its dots.

In [ ]:
def iqr_outliers(data, column):
    """Return the fences and the outlier rows of a column using the IQR rule."""
    q1  = data[column].quantile(0.25)
    q3  = data[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = data[(data[column] < lower) | (data[column] > upper)]
    return lower, upper, outliers

summary_rows = []
for col in continuous_cols:
    lower, upper, outliers = iqr_outliers(df_clean, col)
    summary_rows.append({
        "variable": col,
        "lower_fence": round(lower, 2),
        "upper_fence": round(upper, 2),
        "outlier_count": len(outliers)
    })

pd.DataFrame(summary_rows)

**Explanation**

- The function calculates Q1, Q3 and the IQR, then the two fences.
- `(data[column] < lower) | (data[column] > upper)` selects rows outside the fences. The symbol `|` means **OR**. (We use `&` for **AND**.) Each condition needs its own brackets.
- We loop through all continuous columns and collect one dictionary of results for each, then convert the list of dictionaries into a table.

An `outlier_count` of 0 means that column has no outliers by the IQR rule.

In [ ]:
# Look at the actual rows for one column, for example study_hours
lower, upper, outliers = iqr_outliers(df_clean, "study_hours")
print(f"study_hours fences: {lower:.2f} to {upper:.2f}")
outliers[["student_id", "study_hours", "marks"]]

### What should we do with outliers?

| Situation | Action |
|---|---|
| Clearly an error (age = 200, marks = 150 when the maximum is 100) | Correct it or remove it |
| Genuine but extreme value | Keep it, but use the **median** and **IQR** instead of mean and std |
| Not sure | Keep it, and check results with and without it |

Never delete values just because they look unusual. Always have a reason.

### Sanity check for impossible values

Some values are impossible regardless of statistics. For example marks must be between 0 and 100, and attendance must be between 0 and 100 percent.

In [ ]:
print("Marks outside 0-100       :", ((df_clean["marks"] < 0) | (df_clean["marks"] > 100)).sum())
print("Attendance outside 0-100  :", ((df_clean["attendance"] < 0) | (df_clean["attendance"] > 100)).sum())
print("Negative study hours      :", (df_clean["study_hours"] < 0).sum())
print("Negative AI usage hours   :", (df_clean["ai_usage_hours"] < 0).sum())

## Part 8: Bivariate Analysis (Two Variables Together)

Now we look at **relationships**. The right method depends on the types of the two variables.

| Variable 1 | Variable 2 | Summary | Graph |
|---|---|---|---|
| Continuous | Continuous | Correlation | Scatter plot, heatmap |
| Categorical | Continuous | Group-wise statistics (`groupby`) | Boxplot, bar plot (mean) |
| Categorical | Categorical | Cross table (`crosstab`) | Stacked bar chart, heatmap |

### 8.1 Continuous vs Continuous

#### Scatter plot with a trend line

In Session 5 we drew a scatter plot using pandas. Now we also add a **regression line** that shows the overall trend.

In [ ]:
plt.figure(figsize=(8, 5))
sns.regplot(data=df_clean, x="study_hours", y="marks", scatter_kws={"alpha": 0.7})
plt.title("Study Hours vs Marks")
plt.show()

**Explanation**

- `sns.regplot()` draws the scatter plot and a best-fit straight line (the shaded band around the line shows the uncertainty).
- `scatter_kws={"alpha": 0.7}` makes the dots slightly transparent so overlapping dots are visible.
- A line going **up** means a positive relationship. A line going **down** means a negative relationship. A **flat** line means little or no linear relationship.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, ["study_hours", "attendance", "ai_usage_hours"]):
    sns.regplot(data=df_clean, x=col, y="marks", scatter_kws={"alpha": 0.7}, ax=ax)
    ax.set_title(f"{col} vs marks")

plt.tight_layout()
plt.show()

**Explanation:** `zip(axes, list)` pairs each plot area with one column name, so the loop handles both together.

#### Correlation

**Correlation coefficient (r)** is a number between -1 and +1 that measures the strength and direction of a **linear** relationship between two continuous variables.

| Value of r | Interpretation |
|---|---|
| +0.7 to +1.0 | Strong positive |
| +0.3 to +0.7 | Moderate positive |
| -0.3 to +0.3 | Weak or none |
| -0.7 to -0.3 | Moderate negative |
| -1.0 to -0.7 | Strong negative |

In [ ]:
corr = df_clean[continuous_cols].corr()
corr

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Correlation Heatmap")
plt.show()

**Explanation**

- `.corr()` calculates the correlation of every pair of columns. The result is symmetric and the diagonal is always 1 (every variable is perfectly correlated with itself).
- `sns.heatmap()` colours each cell: red for positive, blue for negative, white for zero.
- `annot=True` writes the number in each cell, `fmt=".2f"` shows two decimals, and `vmin=-1, vmax=1` fixes the colour scale.

**How to read it:** look at the `marks` row. Which variables have the strongest correlation with marks?

#### Correlation with the target variable, sorted

In [ ]:
corr["marks"].drop("marks").sort_values(ascending=False)

**Explanation**

- `corr["marks"]` takes the marks column of the correlation table.
- `.drop("marks")` removes marks' correlation with itself (always 1).
- `.sort_values(ascending=False)` lists the strongest positive relationship first.

### Very important warning

**Correlation does not mean causation.** If two variables move together, it does not prove that one causes the other. There may be a third factor, or it may be coincidence.
Also, correlation only measures **straight-line** relationships. Always look at the scatter plot too.

### 8.2 Categorical vs Continuous

Here we compare a continuous variable **across groups**. The main tool is `groupby()`.

**How `groupby` works:** Split the data into groups, Apply a calculation to each group, and Combine the results into a table.

In [ ]:
# Average marks for each gender
df_clean.groupby("gender")["marks"].mean()

In [ ]:
# Many statistics at once for each grade
df_clean.groupby("grade", observed=True)["marks"].agg(["count", "mean", "median", "std", "min", "max"]).round(2)

**Explanation**

- `groupby("grade")` splits the students into one group per grade.
- `["marks"]` picks the column we want to calculate on.
- `.agg([...])` calculates several statistics for each group.
- `observed=True` tells pandas to show only the categories that actually appear (needed for ordered categories).
- This table also acts as a **check**: students with grade A should have the highest marks.

In [ ]:
# Compare several groups in a loop
for group_col in ["gender", "section", "year", "ai_tool", "complaint"]:
    print("=" * 50)
    print("Marks by", group_col)
    print("=" * 50)
    display(df_clean.groupby(group_col)["marks"].agg(["count", "mean", "median", "std"]).round(2))
    print()

**Explanation**

Here we repeat the same analysis for five different categorical variables. Look at both `mean` and `median`, and also at `count`: a group with only a few students gives an unreliable average.

#### Average of all continuous variables for each year

In [ ]:
df_clean.groupby("year")[continuous_cols].mean().round(2)

**Explanation:** If we do not select one column after `groupby`, pandas calculates the statistic for every column we pass in the list. This gives a quick profile of each year group.

#### Pivot table: two categorical variables and one continuous variable

In [ ]:
pd.pivot_table(
    df_clean,
    values="marks",
    index="section",
    columns="gender",
    aggfunc="mean"
).round(2)

**Explanation**

- `index` becomes the rows, `columns` becomes the columns, and `values` is the continuous variable to summarise.
- `aggfunc="mean"` is the calculation applied in each cell. Try `"median"`, `"count"` or `"max"` too.
- Each cell answers a question like: what is the average marks of **male students in section A**?

#### Boxplot of a continuous variable across groups

Boxplots side by side are the best graph for comparing distributions between groups.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_clean, x="grade", y="marks", order=grade_order, color="lightsteelblue")
plt.title("Marks by Grade")
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
order = df_clean.groupby("ai_tool")["marks"].median().sort_values(ascending=False).index
sns.boxplot(data=df_clean, x="ai_tool", y="marks", order=order, color="lightsteelblue")
plt.title("Marks by AI Tool (sorted by median)")
plt.xticks(rotation=20)
plt.show()

**Explanation**

- `x` is the categorical variable and `y` is the continuous variable. One box is drawn for every category.
- In the second plot, we first calculate the median marks for each tool, sort them, and use that order for the boxes.
- Compare the **median lines** (is one group higher?), the **box heights** (which group is more spread out?) and the **overlap** between boxes (if boxes overlap a lot, the groups are not very different).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, ["gender", "section", "year"]):
    sns.barplot(data=df_clean, x=col, y="marks", color="steelblue", errorbar="sd", ax=ax)
    ax.set_title(f"Average Marks by {col}")

plt.tight_layout()
plt.show()

**Explanation**

- `sns.barplot()` draws the **mean** of `y` for each category as a bar.
- `errorbar="sd"` adds a small black line on each bar showing the standard deviation. Long lines mean the students in that group are very different from each other.
- Small differences in bar height, especially with long error lines, are often just random variation.

### 8.3 Categorical vs Categorical

We use a **cross table** (`crosstab`) which counts how many rows fall into each combination of two categories.

In [ ]:
pd.crosstab(df_clean["gender"], df_clean["grade"])

**Explanation:** Rows are gender, columns are grade and each cell is a count. For example, the cell (Male, A) is the number of male students who got grade A.

#### Percentages are more useful than raw counts

Raw counts are misleading when groups are of different sizes (60 males vs 40 females). We use **percentages within each row**.

In [ ]:
# Row percentages: each ROW adds up to 100 percent
pd.crosstab(df_clean["gender"], df_clean["grade"], normalize="index").mul(100).round(1)

**Explanation**

- `normalize="index"` divides each cell by its **row** total, so every row adds up to 100.
- `normalize="columns"` would divide by the column total instead.
- Reading example: among all male students, what percent got grade A?

#### Adding totals with `margins`

In [ ]:
pd.crosstab(df_clean["section"], df_clean["grade"], margins=True, margins_name="Total")

**Explanation:** `margins=True` adds a `Total` row and a `Total` column so we can check the sums.

#### Stacked bar chart

In [ ]:
ct = pd.crosstab(df_clean["section"], df_clean["grade"], normalize="index").mul(100)

ct.plot(kind="bar", stacked=True, figsize=(8, 5), colormap="viridis")
plt.title("Grade Distribution within each Section (percent)")
plt.ylabel("Percent of students")
plt.xlabel("Section")
plt.legend(title="Grade", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=0)
plt.show()

**Explanation**

- `.plot(kind="bar", stacked=True)` stacks the grade percentages on top of each other so each bar totals 100 percent.
- `bbox_to_anchor=(1.02, 1)` moves the legend outside the plot so it does not cover the bars.
- This graph answers: is the grade mix **similar** in all sections, or does one section have noticeably more of some grade?

In [ ]:
# Heatmap of counts: AI tool vs grade
ct_counts = pd.crosstab(df_clean["ai_tool"], df_clean["grade"])

plt.figure(figsize=(8, 5))
sns.heatmap(ct_counts, annot=True, fmt="d", cmap="Blues")
plt.title("Number of Students: AI Tool vs Grade")
plt.show()

**Explanation**

- A heatmap colours each cell of the crosstab by its count (darker means more students).
- `fmt="d"` shows the numbers as whole integers.

#### Is the relationship real? The Chi-square test (optional, short introduction)

The **Chi-square test of independence** checks whether two categorical variables are related, or whether the pattern could be due to chance.

- **Null hypothesis:** the two variables are independent (no relationship).
- If the **p-value is below 0.05**, we say there is statistical evidence of a relationship.
- With only 100 rows and many categories, the test is not very reliable, so treat the result as a rough guide only.

In [ ]:
from scipy.stats import chi2_contingency

table = pd.crosstab(df_clean["gender"], df_clean["complaint"])
chi2, p_value, dof, expected = chi2_contingency(table)

print("Chi-square statistic:", round(chi2, 3))
print("p-value             :", round(p_value, 4))
print("Degrees of freedom  :", dof)

if p_value < 0.05:
    print("Result: evidence of a relationship between gender and complaint.")
else:
    print("Result: no strong evidence of a relationship between gender and complaint.")

**Explanation**

- `chi2_contingency(table)` returns four things: the test statistic, the p-value, the degrees of freedom and the **expected counts** (what the table would look like if there were no relationship).
- The larger the difference between the real table and the expected table, the smaller the p-value.

## Part 9: Multivariate Analysis and New Features

### 9.1 Three variables in one plot

We can show a **third variable** using colour (`hue`).

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(data=df_clean, x="study_hours", y="marks", hue="gender", style="gender", s=80)
plt.title("Study Hours vs Marks, split by Gender")
plt.show()

**Explanation**

- `hue="gender"` colours the dots by gender, `style="gender"` also changes the dot shape, and `s=80` sets the dot size.
- We can now check whether the pattern (the trend) is the same for both groups.

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(data=df_clean, x="ai_usage_hours", y="marks", hue="year", palette="viridis", s=80)
plt.title("AI Usage Hours vs Marks, coloured by Year")
plt.legend(title="Year")
plt.show()

### 9.2 Pair plot

A **pair plot** draws scatter plots for **every pair** of continuous variables, and a histogram or density curve for each variable on the diagonal. It gives a whole overview in one picture.

In [ ]:
sns.pairplot(
    df_clean[["study_hours", "attendance", "ai_usage_hours", "marks", "gender"]],
    hue="gender",
    diag_kind="kde",
    height=2.4
)
plt.show()

**Explanation**

- We select only 4 continuous columns plus the colouring column, because too many columns make the grid unreadable.
- `diag_kind="kde"` draws a density curve on the diagonal.
- `height=2.4` is the size of each small plot.

### 9.3 Creating new categorical variables from continuous ones (Binning)

Sometimes it helps to convert a continuous variable into groups. This is called **binning** or **discretization**.

| Function | How it creates bins |
|---|---|
| `pd.cut()` | Bins with **our own boundaries** (you decide the cut points) |
| `pd.qcut()` | Bins with **equal number of students** in each (quantile based) |

In [ ]:
# pd.cut: our own boundaries
df_clean["attendance_level"] = pd.cut(
    df_clean["attendance"],
    bins=[0, 75, 90, 100],
    labels=["Low (up to 75)", "Medium (75-90)", "High (90-100)"]
)

# pd.qcut: three equal-sized groups
df_clean["study_level"] = pd.qcut(
    df_clean["study_hours"],
    q=3,
    labels=["Low", "Medium", "High"]
)

print(df_clean["attendance_level"].value_counts())
print()
print(df_clean["study_level"].value_counts())

**Explanation**

- `pd.cut` intervals are open on the left and closed on the right: `(0, 75]`, `(75, 90]`, `(90, 100]`. So a student with exactly 75 percent attendance falls in the first group.
- `pd.qcut(..., q=3)` finds the cut points automatically so each group has about one third of the students.
- `labels` gives readable names to the groups.

Now we can analyse marks across these new groups.

In [ ]:
df_clean.groupby("study_level", observed=True)["marks"].agg(["count", "mean", "median"]).round(2)

In [ ]:
df_clean.groupby("attendance_level", observed=True)["marks"].agg(["count", "mean", "median"]).round(2)

In [ ]:
# Heatmap: average marks for each combination of study level and attendance level
pivot = pd.pivot_table(
    df_clean,
    values="marks",
    index="study_level",
    columns="attendance_level",
    aggfunc="mean",
    observed=True
)

plt.figure(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="YlGnBu")
plt.title("Average Marks: Study Level vs Attendance Level")
plt.show()

**Explanation:** This heatmap combines **two** engineered categories and shows the **average marks** in each combination. It answers: do students with both high study hours and high attendance score the best?

### 9.4 A simple yes/no feature: does the student use AI?

In [ ]:
df_clean["uses_ai"] = np.where(df_clean["ai_tool"] == "No Tool", "No", "Yes")

print(df_clean["uses_ai"].value_counts())
print()
df_clean.groupby("uses_ai")[["marks", "study_hours", "attendance", "ai_usage_hours"]].mean().round(2)

**Explanation**

- `np.where(condition, value_if_true, value_if_false)` is like an Excel `IF` formula applied to the whole column.
- Be careful: only 7 students use no tool, so any comparison with that small group is **not reliable**.

## Part 10: Reusable EDA Function, Insights and Practice

### 10.1 One function to profile any DataFrame

Now we combine what we learned into one function that gives a quick profile of any dataset. Real analysts keep such functions handy.

In [ ]:
def quick_eda(data):
    """Print a compact EDA report for any DataFrame."""
    print("DATASET SHAPE:", data.shape)
    print()

    print("MISSING VALUES:")
    miss = data.isnull().sum()
    print(miss[miss > 0] if miss.sum() > 0 else "None")
    print()

    print("DUPLICATE ROWS:", data.duplicated().sum())
    print()

    num_cols = data.select_dtypes(include="number").columns.tolist()
    cat_cols = data.select_dtypes(exclude="number").columns.tolist()

    print("NUMERIC COLUMNS    :", num_cols)
    print("CATEGORICAL COLUMNS:", cat_cols)
    print()

    print("NUMERIC SUMMARY:")
    display(data[num_cols].describe().T.round(2))
    print()

    print("CATEGORICAL SUMMARY:")
    display(data[cat_cols].astype("object").describe().T)

quick_eda(df)

**Explanation**

- `select_dtypes(include="number")` automatically picks all numeric columns and `exclude="number"` picks everything else. This is a quick way of separating the variable types by data type.
- Remember: this automatic method would treat `year` as numeric. That is why we also defined our own lists by **meaning** earlier.
- We can run `quick_eda(any_dataframe)` on any future dataset.

### 10.2 Saving the cleaned data (optional)

We can save `df_clean` to Google Drive so that we can use it directly in future sessions without repeating the cleaning steps.

In [ ]:
df_clean.to_csv("/content/drive/MyDrive/Colab Notebooks/Data/academic_data_clean.csv", index=False)
print("Cleaned file saved.")

**Explanation**

- `to_csv(path, index=False)` writes the table to a CSV file. `index=False` stops pandas from saving the row numbers as an extra column.
- The file is saved in the same Data folder as `academic_data.csv`, with a new name, so the original is untouched.

### 10.3 Writing your own insights

The last step of EDA is to **write down what you found** in plain language. Fill in the table below in your own words by looking at the outputs above.

| Question | Your finding |
|---|---|
| How many students and variables are there? | |
| Which variable had missing values and how did we handle it? | |
| Which continuous variables are skewed? | |
| Does any continuous variable have outliers? | |
| Which variable has the strongest correlation with marks? | |
| Do marks differ across gender, section, year or AI tool? | |
| Is the grade mix similar across sections? | |
| Which findings are unreliable because the groups are small? | |

### 10.4 Practice Exercises

Try these on your own. Each exercise uses a concept from today.

**Easy**

1. Find the mean, median and standard deviation of `attendance`.
2. Count how many students are in each `section` and show the percentages.
3. Draw a histogram of `study_hours` with 8 bins.

**Medium**

4. Find the average `ai_usage_hours` for every `ai_tool`.
5. Draw a boxplot of `study_hours` for each `year`.
6. Create a crosstab of `year` and `complaint` using row percentages.

**Challenging**

7. Find the outliers in `ai_usage_hours` using the IQR rule and display those students.
8. Create a new column `performance` using `pd.cut` on `marks` with the bins 0-50, 50-70, 70-100 and labels Low, Medium, High. Then check how `performance` varies across `section`.
9. Which two continuous variables have the highest correlation with each other (apart from a variable with itself)? Show a scatter plot of that pair.

### 10.5 Cheat Sheet of Today's Commands

| Task | Code |
|---|---|
| Missing values per column | `df.isnull().sum()` |
| Duplicate rows | `df.duplicated().sum()` |
| Fill missing values | `df["col"].fillna("value")` |
| Replace values | `df["col"].replace("old", "new")` |
| Number of unique values | `df.nunique()` |
| Mean, median, mode | `.mean()`, `.median()`, `.mode()` |
| Std, variance | `.std()`, `.var()` |
| Percentiles | `.quantile([0.25, 0.5, 0.75])` |
| Skewness, kurtosis | `.skew()`, `.kurt()` |
| Frequency table | `df["col"].value_counts()` and `value_counts(normalize=True)` |
| Group-wise statistics | `df.groupby("cat")["num"].agg(["mean", "median"])` |
| Cross table | `pd.crosstab(a, b, normalize="index")` |
| Pivot table | `pd.pivot_table(df, values=, index=, columns=, aggfunc=)` |
| Correlation | `df[cols].corr()` |
| Histogram | `sns.histplot(data=df, x="col", kde=True)` |
| Boxplot | `sns.boxplot(data=df, x="cat", y="num")` |
| Bar chart of counts | `sns.countplot(data=df, x="cat")` |
| Scatter plot with line | `sns.regplot(data=df, x="a", y="b")` |
| Heatmap | `sns.heatmap(table, annot=True)` |
| Bins | `pd.cut(...)` and `pd.qcut(...)` |

### Next Session

In the next session we will build on this EDA and move towards hypothesis testing and statistical comparison between groups.